In [ ]:
!pip install -q flask boto3 "moto[s3,dynamodb]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 85.8 MB/s eta 0:00:00


In [ ]:
import os, io, re, json, uuid, base64, hashlib, mimetypes, threading
from collections import defaultdict
from datetime import datetime, timezone
from decimal import Decimal
from urllib.parse import quote, quote_plus, unquote, unquote_plus
import boto3
from boto3.dynamodb.conditions import Key, Attr
from botocore.exceptions import ClientError

MODE         = 'mock'     # 'mock'  or  'aws'
AUTO_TRIGGER = True       # Colab simulates the S3 event. On real AWS use a real S3 trigger and set this to False.
TABLE        = 'FileIndex'
MAX_BYTES    = 5 * 1024 * 1024

if MODE == 'mock':
    from moto import mock_aws
    os.environ['AWS_ACCESS_KEY_ID'] = 'testing'
    os.environ['AWS_SECRET_ACCESS_KEY'] = 'testing'
    REGION = 'us-east-1'
    BUCKET = 'file-storage-' + uuid.uuid4().hex[:8]
    _mock = mock_aws(); _mock.start()
    aws_kwargs = {'region_name': REGION}
else:
    from google.colab import userdata
    REGION = userdata.get('AWS_REGION')
    BUCKET = userdata.get('S3_BUCKET')
    aws_kwargs = dict(region_name=REGION,
                      aws_access_key_id=userdata.get('AWS_ACCESS_KEY_ID'),
                      aws_secret_access_key=userdata.get('AWS_SECRET_ACCESS_KEY'))

s3  = boto3.client('s3', **aws_kwargs)
ddb = boto3.resource('dynamodb', **aws_kwargs)

def ensure_resources():
    try:
        s3.head_bucket(Bucket=BUCKET)
    except ClientError:
        if REGION == 'us-east-1':
            s3.create_bucket(Bucket=BUCKET)
        else:
            s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={'LocationConstraint': REGION})
    if TABLE not in [t.name for t in ddb.tables.all()]:
        # global secondary index on sha256 = instant "which files share this hash?" lookup
        ddb.create_table(TableName=TABLE,
                         KeySchema=[{'AttributeName': 'file_id', 'KeyType': 'HASH'}],
                         AttributeDefinitions=[{'AttributeName': 'file_id', 'AttributeType': 'S'},
                                               {'AttributeName': 'sha256', 'AttributeType': 'S'}],
                         GlobalSecondaryIndexes=[{'IndexName': 'sha256-index',
                                                  'KeySchema': [{'AttributeName': 'sha256', 'KeyType': 'HASH'}],
                                                  'Projection': {'ProjectionType': 'ALL'}}],
                         BillingMode='PAY_PER_REQUEST').wait_until_exists()
    return ddb.Table(TABLE)

table = ensure_resources()
print(f'Mode: {MODE} | S3 bucket: {BUCKET} | DynamoDB table: {TABLE} | Region: {REGION}')

Mode: mock | S3 bucket: file-storage-774e98e7 | DynamoDB table: FileIndex | Region: us-east-1


In [ ]:
def human_size(n):
    n = float(n)
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if n < 1024 or unit == 'TB':
            return f'{n:.0f} {unit}' if unit == 'B' else f'{n:.2f} {unit}'
        n /= 1024

def hash_object(bucket, key):
    h = hashlib.sha256()
    body = s3.get_object(Bucket=bucket, Key=key)['Body']
    for chunk in body.iter_chunks(chunk_size=1024 * 1024):
        h.update(chunk)
    return h.hexdigest()

def make_s3_event(key, size):
    # same structure AWS sends to Lambda when an object is created in S3
    return {'Records': [{'eventSource': 'aws:s3', 'eventName': 'ObjectCreated:Put',
                         's3': {'bucket': {'name': BUCKET},
                                'object': {'key': quote_plus(key), 'size': size}}}]}

def hash_handler(event, context=None):
    processed = 0
    for rec in event.get('Records', []):
        bucket = rec['s3']['bucket']['name']
        key = unquote_plus(rec['s3']['object']['key'])
        if not key.startswith('files/'):                 # ignore trash/ and anything else
            continue
        file_id, name = key.split('/')[1], key.split('/')[-1]
        head = s3.head_object(Bucket=bucket, Key=key)
        meta = head.get('Metadata', {})
        table.put_item(Item={
            'file_id': file_id, 'name': unquote(meta.get('original-name', name)), 's3_key': key,
            'size_bytes': int(head['ContentLength']), 'sha256': hash_object(bucket, key),
            'content_type': head.get('ContentType', 'application/octet-stream'),
            'uploaded_at': meta.get('uploaded-at') or head['LastModified'].astimezone(timezone.utc).isoformat(),
            'status': 'ACTIVE'})
        processed += 1
    return {'processed': processed}

In [ ]:
def _json_default(o):
    return int(o) if isinstance(o, Decimal) else str(o)

def response(code, payload):
    return {'statusCode': code, 'headers': {'Content-Type': 'application/json'},
            'body': json.dumps(payload, default=_json_default)}

def now_iso():
    return datetime.now(timezone.utc).isoformat()          # includes microseconds -> exact upload order

def scan_status(status):
    kw = dict(FilterExpression=Attr('status').eq(status))
    resp = table.scan(**kw)
    items = resp['Items']
    while 'LastEvaluatedKey' in resp:
        resp = table.scan(ExclusiveStartKey=resp['LastEvaluatedKey'], **kw)
        items += resp['Items']
    for i in items:
        i['size_bytes'] = int(i['size_bytes'])
        i['size_readable'] = human_size(i['size_bytes'])
    return items

def analyse(items):
    # group active files by hash; the oldest upload in each group is the "original"
    by_hash = defaultdict(list)
    for it in items:
        by_hash[it['sha256']].append(it)
    groups = []
    for sha, lst in by_hash.items():
        lst.sort(key=lambda x: (x['uploaded_at'], x['name']))
        for idx, it in enumerate(lst):
            it['group_size'] = len(lst)
            it['is_duplicate'] = idx > 0
            it['original_name'] = lst[0]['name']
        if len(lst) > 1:
            size = lst[0]['size_bytes']
            groups.append({'sha256': sha, 'size_bytes': size, 'size_readable': human_size(size),
                           'count': len(lst), 'wasted_bytes': size * (len(lst) - 1),
                           'wasted_readable': human_size(size * (len(lst) - 1)),
                           'copies': [{'file_id': c['file_id'], 'name': c['name'],
                                       'uploaded_at': c['uploaded_at'], 'is_original': k == 0}
                                      for k, c in enumerate(lst)]})
    groups.sort(key=lambda g: g['wasted_bytes'], reverse=True)
    return by_hash, groups

def do_trash(ids):
    moved = 0
    for fid in ids:
        item = table.get_item(Key={'file_id': fid}).get('Item')
        if not item or item['status'] != 'ACTIVE':
            continue
        old = item['s3_key']
        new = 'trash/' + old[len('files/'):]
        s3.copy_object(Bucket=BUCKET, CopySource={'Bucket': BUCKET, 'Key': old}, Key=new)
        s3.delete_object(Bucket=BUCKET, Key=old)
        table.update_item(Key={'file_id': fid},
                          UpdateExpression='SET #s = :t, #k = :nk, #o = :ok, #ts = :ts',
                          ExpressionAttributeNames={'#s': 'status', '#k': 's3_key', '#o': 'original_key', '#ts': 'trashed_at'},
                          ExpressionAttributeValues={':t': 'TRASHED', ':nk': new, ':ok': old, ':ts': now_iso()})
        moved += 1
    return moved

def handle_upload(body):
    name = os.path.basename(body.get('file_name') or '').strip()
    if not name:
        return response(400, {'error': 'File name is missing.'})
    try:
        raw = base64.b64decode(body.get('file_b64') or '', validate=True)
    except Exception:
        return response(400, {'error': 'File could not be read.'})
    if not raw:
        return response(400, {'error': f'{name} is empty.'})
    if len(raw) > MAX_BYTES:
        return response(400, {'error': f'{name} is too large (max 5 MB in this demo).'})

    file_id = uuid.uuid4().hex[:10]
    key = f"files/{file_id}/{re.sub(r'[^A-Za-z0-9._-]', '_', name)[:80]}"
    s3.put_object(Bucket=BUCKET, Key=key, Body=raw,
                  ContentType=mimetypes.guess_type(name)[0] or 'application/octet-stream',
                  Metadata={'original-name': quote(name, safe=''), 'uploaded-at': now_iso()})
    if AUTO_TRIGGER:
        hash_handler(make_s3_event(key, len(raw)))             # S3 event -> hashing function

    item = table.get_item(Key={'file_id': file_id}).get('Item')
    if not item:
        return response(200, {'file_id': file_id, 'name': name, 'status': 'PENDING'})
    same = table.query(IndexName='sha256-index', KeyConditionExpression=Key('sha256').eq(item['sha256']))['Items']
    same = sorted([s for s in same if s['status'] == 'ACTIVE'], key=lambda x: (x['uploaded_at'], x['name']))
    dup_of = same[0]['name'] if same and same[0]['file_id'] != file_id else ''
    return response(200, {'file_id': file_id, 'name': name, 'size_readable': human_size(len(raw)),
                          'sha256': item['sha256'], 'duplicate_of': dup_of, 'copies': len(same)})

def handle_overview(body):
    active, trashed = scan_status('ACTIVE'), scan_status('TRASHED')
    by_hash, groups = analyse(active)
    total = sum(i['size_bytes'] for i in active)
    wasted = sum(g['wasted_bytes'] for g in groups)
    stats = {'total_files': len(active), 'total_bytes': total, 'total_readable': human_size(total),
             'unique_files': len(by_hash), 'duplicate_files': len(active) - len(by_hash),
             'duplicate_groups': len(groups), 'wasted_bytes': wasted, 'wasted_readable': human_size(wasted),
             'wasted_pct': round(wasted / total * 100, 1) if total else 0,
             'trash_files': len(trashed), 'trash_readable': human_size(sum(i['size_bytes'] for i in trashed))}
    active.sort(key=lambda x: x['uploaded_at'], reverse=True)
    trashed.sort(key=lambda x: x.get('trashed_at', ''), reverse=True)
    return response(200, {'stats': stats, 'groups': groups, 'items': active, 'trash': trashed})

def handle_cleanup(body):
    # automatic cleanup: keep ONE copy per group (oldest or newest), trash the rest
    strategy = body.get('strategy', 'oldest')
    by_hash, groups = analyse(scan_status('ACTIVE'))
    plan, saved = [], 0
    for lst in by_hash.values():
        if len(lst) < 2:
            continue
        keep = lst[-1] if strategy == 'newest' else lst[0]
        for it in lst:
            if it['file_id'] != keep['file_id']:
                plan.append({'file_id': it['file_id'], 'name': it['name'], 'size_readable': it['size_readable'],
                             'size_bytes': it['size_bytes'], 'keeping': keep['name']})
                saved += it['size_bytes']
    if not body.get('dry_run', True):
        do_trash([p['file_id'] for p in plan])
    return response(200, {'dry_run': body.get('dry_run', True), 'files': len(plan),
                          'saved_bytes': saved, 'saved_readable': human_size(saved), 'plan': plan})

def handle_trash(body):
    ids = set(body.get('file_ids') or [])
    if not ids:
        return response(400, {'error': 'No files selected.'})
    by_hash, _ = analyse(scan_status('ACTIVE'))
    active_ids = {i['file_id'] for lst in by_hash.values() for i in lst}
    if not ids <= active_ids:
        return response(404, {'error': 'Some selected files no longer exist. Refresh and try again.'})
    for lst in by_hash.values():
        group_ids = {i['file_id'] for i in lst}
        if len(lst) >= 2 and group_ids <= ids:
            return response(400, {'error': f'You selected every copy of "{lst[0]["name"]}". Keep at least one.'})
    size = sum(i['size_bytes'] for lst in by_hash.values() for i in lst if i['file_id'] in ids)
    moved = do_trash(ids)
    return response(200, {'trashed': moved, 'freed_readable': human_size(size)})

def handle_restore(body):
    restored = 0
    for fid in body.get('file_ids') or []:
        item = table.get_item(Key={'file_id': fid}).get('Item')
        if not item or item['status'] != 'TRASHED':
            continue
        s3.copy_object(Bucket=BUCKET, CopySource={'Bucket': BUCKET, 'Key': item['s3_key']}, Key=item['original_key'])
        s3.delete_object(Bucket=BUCKET, Key=item['s3_key'])
        table.update_item(Key={'file_id': fid},
                          UpdateExpression='SET #s = :a, #k = :k REMOVE #ts, #o',
                          ExpressionAttributeNames={'#s': 'status', '#k': 's3_key', '#ts': 'trashed_at', '#o': 'original_key'},
                          ExpressionAttributeValues={':a': 'ACTIVE', ':k': item['original_key']})
        restored += 1
    return response(200, {'restored': restored})

def handle_purge(body):
    # permanently delete; with no ids given = empty the whole trash
    ids = body.get('file_ids') or [i['file_id'] for i in scan_status('TRASHED')]
    deleted = 0
    for fid in ids:
        item = table.get_item(Key={'file_id': fid}).get('Item')
        if item and item['status'] == 'TRASHED':
            s3.delete_object(Bucket=BUCKET, Key=item['s3_key'])
            table.delete_item(Key={'file_id': fid})
            deleted += 1
    return response(200, {'deleted': deleted})

def handle_file(body):
    item = table.get_item(Key={'file_id': body.get('file_id', '')}).get('Item')
    if not item:
        return response(404, {'error': 'File not found.'})
    data = s3.get_object(Bucket=BUCKET, Key=item['s3_key'])['Body'].read()
    safe = re.sub(r'[^A-Za-z0-9._ -]', '_', item['name'])
    return {'statusCode': 200, 'isBase64Encoded': True, 'body': base64.b64encode(data).decode(),
            'headers': {'Content-Type': item.get('content_type', 'application/octet-stream'),
                        'Content-Disposition': f'attachment; filename="{safe}"'}}

ROUTES = {'/upload': handle_upload, '/overview': handle_overview, '/cleanup': handle_cleanup,
          '/trash': handle_trash, '/restore': handle_restore, '/purge': handle_purge, '/file': handle_file}

def lambda_handler(event, context=None):
    route = ROUTES.get(event.get('path', ''))
    if route is None:
        return response(404, {'error': 'Unknown route'})
    body = event.get('body') or {}
    if isinstance(body, str):
        try:
            body = json.loads(body)
        except json.JSONDecodeError:
            return response(400, {'error': 'Body must be valid JSON.'})
    return route(body)

In [ ]:
from flask import Flask, request, Response

app = Flask(__name__)

def to_flask(res):
    headers = dict(res['headers'])
    mimetype = headers.pop('Content-Type')
    data = base64.b64decode(res['body']) if res.get('isBase64Encoded') else res['body']
    return Response(data, status=res['statusCode'], mimetype=mimetype, headers=headers)

@app.post('/api/<path:action>')
def api(action):
    body = json.dumps(request.get_json(silent=True) or {})
    return to_flask(lambda_handler({'path': '/' + action, 'body': body}, None))

@app.get('/api/file/<file_id>')
def api_file(file_id):
    return to_flask(lambda_handler({'path': '/file', 'body': json.dumps({'file_id': file_id})}, None))

In [ ]:
HTML_PAGE = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Duplicate File Cleanup</title>
<style>
  * { box-sizing: border-box; }
  body { font-family: Arial, sans-serif; background: #f3f4f8; margin: 0; color: #1f2937; }
  header { background: #4338ca; color: #fff; padding: 18px 24px; }
  header h1 { margin: 0; font-size: 22px; }
  header p { margin: 4px 0 0; font-size: 13px; opacity: .9; }
  main { max-width: 1100px; margin: 20px auto; padding: 0 16px; }
  .card { background: #fff; border-radius: 12px; padding: 18px; box-shadow: 0 2px 8px rgba(0,0,0,.08); margin-bottom: 20px; }
  h3 { margin: 0 0 10px; }
  .stats { display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 14px; margin-bottom: 20px; }
  .stat { background: #fff; border-radius: 12px; padding: 14px; text-align: center; box-shadow: 0 2px 8px rgba(0,0,0,.08); }
  .stat b { display: block; font-size: 22px; color: #4338ca; } .stat.warn b { color: #d97706; } .stat.ok b { color: #16a34a; }
  .stat span { font-size: 12px; color: #6b7280; }
  .usage { height: 22px; background: #c7d2fe; border-radius: 11px; overflow: hidden; margin: 8px 0 4px; }
  .usage div { height: 100%; background: #f59e0b; }
  #drop { border: 2px dashed #818cf8; border-radius: 12px; padding: 26px; text-align: center; background: #eef2ff; cursor: pointer; }
  #drop.over { background: #e0e7ff; }
  button, a.btn { background: #4338ca; color: #fff; border: none; padding: 8px 15px; border-radius: 8px; font-size: 13px; cursor: pointer; text-decoration: none; display: inline-block; }
  button:hover { background: #3730a3; }
  button.danger { background: #dc2626; } button.ghost { background: #e5e7eb; color: #111; } button.green { background: #16a34a; }
  button:disabled { background: #cbd5e1; cursor: not-allowed; }
  button.sm { padding: 4px 10px; font-size: 12px; }
  select, input[type=text] { padding: 7px; border: 1px solid #cbd5e1; border-radius: 8px; font-size: 13px; }
  .row { display: flex; gap: 10px; align-items: center; flex-wrap: wrap; margin-bottom: 10px; }
  .small { font-size: 12px; color: #6b7280; }
  table { width: 100%; border-collapse: collapse; font-size: 13px; }
  th, td { text-align: left; padding: 7px 6px; border-bottom: 1px solid #e5e7eb; vertical-align: middle; }
  th { background: #f8fafc; }
  .group { border: 1px solid #e5e7eb; border-radius: 10px; padding: 12px; margin-bottom: 12px; }
  .group h4 { margin: 0 0 8px; font-size: 14px; }
  .badge { padding: 2px 9px; border-radius: 10px; font-size: 11px; font-weight: bold; color: #fff; }
  .b-orig { background: #16a34a; } .b-dup { background: #d97706; } .b-uni { background: #64748b; }
  .hash { font-family: monospace; font-size: 11px; color: #6b7280; }
  .msg { font-size: 13px; padding: 4px 0; } .msg.dup { color: #b45309; font-weight: bold; } .msg.err { color: #dc2626; }
  .plan { background: #f8fafc; border-radius: 8px; padding: 10px; font-size: 13px; margin-top: 8px; max-height: 220px; overflow: auto; }
  #toast { position: fixed; bottom: 18px; left: 50%; transform: translateX(-50%); background: #1e293b; color: #fff; padding: 9px 16px; border-radius: 8px; font-size: 13px; display: none; z-index: 10; }
  .overflow { overflow-x: auto; }
</style>
</head>
<body>
<header>
  <h1>Serverless Duplicate File Detection &amp; Cleanup</h1>
  <p>Upload to S3 &rarr; event triggers hashing function &rarr; SHA-256 index in DynamoDB &rarr; safe cleanup</p>
</header>
<main>
  <div class="stats">
    <div class="stat"><b id="s-files">0</b><span>Files</span></div>
    <div class="stat"><b id="s-total">0 B</b><span>Storage used</span></div>
    <div class="stat warn"><b id="s-dups">0</b><span>Duplicate files</span></div>
    <div class="stat warn"><b id="s-wasted">0 B</b><span>Wasted by duplicates</span></div>
    <div class="stat ok"><b id="s-trash">0</b><span>In trash</span></div>
  </div>

  <div class="card">
    <h3>Storage health</h3>
    <div class="usage"><div id="bar" style="width:0%"></div></div>
    <div class="small" id="barlabel"></div>
  </div>

  <div class="card">
    <div id="drop"><b>Drag &amp; drop files here</b> or click to choose<br>
      <span class="small">Any file type, max 5 MB each. Upload the same file twice to see detection.</span>
      <input type="file" id="file" multiple style="display:none"></div>
    <div id="uplog" style="margin-top:10px"></div>
  </div>

  <div class="card">
    <h3>Duplicate groups</h3>
    <div class="row">
      <b>Auto-clean:</b>
      <select id="strategy"><option value="oldest">Keep the oldest copy</option><option value="newest">Keep the newest copy</option></select>
      <button class="ghost" onclick="autoClean(true)">Preview</button>
      <button class="green" id="runclean" onclick="autoClean(false)">Clean up now</button>
      <span class="small">Removed files go to the trash first, so you can restore them.</span>
    </div>
    <div id="plan"></div>
    <hr style="border:none;border-top:1px solid #e5e7eb;margin:14px 0">
    <div id="groups"></div>
    <div class="row">
      <button id="trashsel" onclick="trashSelected()">Move selected to trash</button>
      <span id="selinfo" class="small"></span>
    </div>
  </div>

  <div class="card">
    <h3>All files</h3>
    <div class="row">
      <select id="flt" onchange="renderFiles()"><option value="all">All files</option><option value="dup">Duplicates only</option><option value="uni">Unique only</option></select>
      <input type="text" id="q" placeholder="Search name" oninput="renderFiles()">
    </div>
    <div class="overflow"><table>
      <thead><tr><th>Name</th><th>Size</th><th>SHA-256</th><th>Status</th><th>Uploaded (UTC)</th><th></th></tr></thead>
      <tbody id="files"></tbody>
    </table></div>
  </div>

  <div class="card">
    <div class="row"><h3 style="margin:0">Trash</h3><button class="danger sm" id="empty">Empty trash</button></div>
    <div class="overflow"><table>
      <thead><tr><th>Name</th><th>Size</th><th>Trashed (UTC)</th><th></th></tr></thead>
      <tbody id="trash"></tbody>
    </table></div>
  </div>
</main>
<div id="toast"></div>

<script>
const $ = id => document.getElementById(id);
let D = {stats: {}, groups: [], items: [], trash: []}, sel = new Set();

function el(tag, attrs, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) {
    if (k === 'class') e.className = v; else if (k === 'style') e.style.cssText = v;
    else if (k.startsWith('on') || k === 'checked' || k === 'value' || k === 'disabled') e[k] = v;
    else e.setAttribute(k, v);
  }
  kids.forEach(c => { if (c !== null && c !== undefined) e.append(c); });
  return e;
}
function human(n) { const u = ['B', 'KB', 'MB', 'GB']; let i = 0; while (n >= 1024 && i < 3) { n /= 1024; i++; } return (i ? n.toFixed(2) : n.toFixed(0)) + ' ' + u[i]; }
function ts(s) { return (s || '').slice(0, 19).replace('T', ' '); }
function toast(m) { const t = $('toast'); t.textContent = m; t.style.display = 'block'; clearTimeout(t._h); t._h = setTimeout(() => t.style.display = 'none', 3500); }
function twoClick(btn, label, action) {
  btn.onclick = () => {
    if (btn.dataset.sure) { action(); return; }
    btn.dataset.sure = 1; btn.textContent = 'Sure?';
    setTimeout(() => { delete btn.dataset.sure; btn.textContent = label; }, 2500);
  };
}
async function api(action, body) {
  const r = await fetch('api/' + action, {method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify(body || {})});
  let data = {}; try { data = await r.json(); } catch (e) {}
  return {ok: r.ok, data: data};
}
function readFile(f) { return new Promise(res => { const r = new FileReader(); r.onload = () => res(r.result.split(',')[1]); r.readAsDataURL(f); }); }

// ---------- upload ----------
const drop = $('drop');
drop.onclick = () => $('file').click();
drop.ondragover = e => { e.preventDefault(); drop.classList.add('over'); };
drop.ondragleave = () => drop.classList.remove('over');
drop.ondrop = e => { e.preventDefault(); drop.classList.remove('over'); upload(e.dataTransfer.files); };
$('file').onchange = e => upload(e.target.files);

async function upload(files) {
  const log = $('uplog'); log.replaceChildren();
  for (const f of [...files]) {
    if (f.size > 5 * 1024 * 1024) { log.append(el('div', {class: 'msg err'}, f.name + ': too large (max 5 MB)')); continue; }
    const r = await api('upload', {file_name: f.name, file_b64: await readFile(f)});
    if (!r.ok) { log.append(el('div', {class: 'msg err'}, f.name + ': ' + r.data.error)); continue; }
    const d = r.data;
    if (d.duplicate_of) log.append(el('div', {class: 'msg dup'}, f.name + ' (' + d.size_readable + ') is a DUPLICATE of "' + d.duplicate_of + '" - ' + d.copies + ' identical copies now'));
    else log.append(el('div', {class: 'msg'}, f.name + ' (' + d.size_readable + ') uploaded - unique'));
  }
  $('file').value = ''; load();
}

// ---------- load + render ----------
async function load() {
  const r = await api('overview'); D = r.data;
  sel = new Set(D.items.filter(i => i.is_duplicate).map(i => i.file_id));   // pre-select the duplicates
  render();
}
function render() { renderStats(); renderGroups(); renderFiles(); renderTrash(); }

function renderStats() {
  const s = D.stats;
  $('s-files').textContent = s.total_files; $('s-total').textContent = s.total_readable;
  $('s-dups').textContent = s.duplicate_files; $('s-wasted').textContent = s.wasted_readable; $('s-trash').textContent = s.trash_files;
  $('bar').style.width = s.wasted_pct + '%';
  $('barlabel').textContent = s.total_files ? s.wasted_pct + '% of your storage (' + s.wasted_readable + ') is taken up by duplicate copies. ' + s.unique_files + ' unique file(s), ' + s.duplicate_groups + ' duplicate group(s).' : 'No files yet.';
  $('runclean').disabled = !s.duplicate_files;
}

function renderGroups() {
  const box = $('groups'); box.replaceChildren();
  if (!D.groups.length) box.append(el('div', {class: 'small'}, 'No duplicates found. Upload the same file twice to test.'));
  D.groups.forEach(g => {
    const card = el('div', {class: 'group'});
    card.append(el('h4', {}, g.count + ' identical copies - ' + g.size_readable + ' each - wasting ' + g.wasted_readable + '  ', el('span', {class: 'hash'}, g.sha256.slice(0, 16) + '...')));
    const t = el('table');
    g.copies.forEach(c => {
      const tr = el('tr');
      tr.append(el('td', {style: 'width:30px'}, el('input', {type: 'checkbox', checked: sel.has(c.file_id), onchange: e => { e.target.checked ? sel.add(c.file_id) : sel.delete(c.file_id); updateBar(); }})));
      tr.append(el('td', {}, c.name), el('td', {}, el('span', {class: 'badge ' + (c.is_original ? 'b-orig' : 'b-dup')}, c.is_original ? 'ORIGINAL (oldest)' : 'DUPLICATE')));
      tr.append(el('td', {class: 'small'}, ts(c.uploaded_at)), el('td', {}, el('a', {href: 'api/file/' + c.file_id, target: '_blank'}, 'Download')));
      t.append(tr);
    });
    card.append(t); box.append(card);
  });
  updateBar();
}
function updateBar() {
  const byId = Object.fromEntries(D.items.map(i => [i.file_id, i]));
  let bytes = 0; sel.forEach(id => { if (byId[id]) bytes += byId[id].size_bytes; });
  const allSel = D.groups.find(g => g.copies.every(c => sel.has(c.file_id)));
  $('trashsel').disabled = !sel.size || !!allSel;
  $('selinfo').textContent = allSel ? 'Every copy of "' + allSel.copies[0].name + '" is ticked. Untick at least one.' : sel.size + ' file(s) selected - frees ' + human(bytes);
}
async function trashSelected() {
  const r = await api('trash', {file_ids: [...sel]});
  toast(r.ok ? 'Moved ' + r.data.trashed + ' file(s) to trash, freed ' + r.data.freed_readable : r.data.error);
  $('plan').replaceChildren(); load();
}
async function autoClean(dry) {
  const r = await api('cleanup', {strategy: $('strategy').value, dry_run: dry});
  const d = r.data, box = $('plan'); box.replaceChildren();
  if (!d.files) { box.append(el('div', {class: 'small'}, 'Nothing to clean: no duplicates.')); return; }
  if (!dry) { toast('Moved ' + d.files + ' duplicate(s) to trash, freed ' + d.saved_readable); load(); return; }
  const p = el('div', {class: 'plan'}, el('b', {}, 'Preview: ' + d.files + ' file(s) would move to trash, freeing ' + d.saved_readable));
  d.plan.forEach(x => p.append(el('div', {}, '- ' + x.name + ' (' + x.size_readable + ')  [keeping: ' + x.keeping + ']')));
  box.append(p);
}

function renderFiles() {
  const f = $('flt').value, q = $('q').value.toLowerCase();
  const tb = $('files'); tb.replaceChildren();
  const shown = D.items.filter(i => (f === 'all' || (f === 'dup' && i.group_size > 1) || (f === 'uni' && i.group_size === 1)) && (!q || i.name.toLowerCase().includes(q)));
  shown.forEach(i => {
    const tr = el('tr');
    let status = el('span', {class: 'badge b-uni'}, 'Unique');
    if (i.group_size > 1) status = el('span', {class: 'badge ' + (i.is_duplicate ? 'b-dup' : 'b-orig')}, i.is_duplicate ? 'Duplicate of ' + i.original_name : 'Original (' + i.group_size + ' copies)');
    tr.append(el('td', {}, i.name), el('td', {}, i.size_readable), el('td', {class: 'hash'}, i.sha256.slice(0, 12) + '...'), el('td', {}, status), el('td', {class: 'small'}, ts(i.uploaded_at)));
    const act = el('td'); act.append(el('a', {href: 'api/file/' + i.file_id, target: '_blank', style: 'margin-right:8px;font-size:12px'}, 'Download'));
    const b = el('button', {class: 'danger sm'}, 'Trash'); twoClick(b, 'Trash', async () => { const r = await api('trash', {file_ids: [i.file_id]}); toast(r.ok ? 'Moved to trash' : r.data.error); load(); });
    act.append(b); tr.append(act); tb.append(tr);
  });
  if (!shown.length) tb.append(el('tr', {}, el('td', {colspan: 6, class: 'small'}, 'No files to show.')));
}

function renderTrash() {
  const tb = $('trash'); tb.replaceChildren();
  D.trash.forEach(i => {
    const tr = el('tr'); tr.append(el('td', {}, i.name), el('td', {}, i.size_readable), el('td', {class: 'small'}, ts(i.trashed_at)));
    const act = el('td');
    act.append(el('button', {class: 'green sm', onclick: async () => { await api('restore', {file_ids: [i.file_id]}); toast('Restored ' + i.name); load(); }}, 'Restore'), ' ');
    const del = el('button', {class: 'danger sm'}, 'Delete forever');
    twoClick(del, 'Delete forever', async () => { await api('purge', {file_ids: [i.file_id]}); load(); });
    act.append(del); tr.append(act); tb.append(tr);
  });
  if (!D.trash.length) tb.append(el('tr', {}, el('td', {colspan: 4, class: 'small'}, 'Trash is empty.')));
  const e = $('empty'); e.disabled = !D.trash.length;
  twoClick(e, 'Empty trash', async () => { const r = await api('purge', {}); toast('Permanently deleted ' + r.data.deleted + ' file(s)'); load(); });
}
load();
</script>
</body>
</html>
"""

@app.get('/')
def home():
    return Response(HTML_PAGE, mimetype='text/html')

## 7. Test with sample files (optional, good screenshots for your report)
Uploads 9 files (some are exact duplicates under different names), shows the detection result, previews and runs the automatic cleanup, then **restores everything** so you can try the cleanup yourself in the web app. Skip this cell to start empty.

In [ ]:
client = app.test_client()

def call(action, **body):
    r = client.post('/api/' + action, json=body)
    return r.status_code, r.get_json()

pdf   = b'%PDF-1.4 ' + os.urandom(60_000)
photo = os.urandom(200_000)
text  = ('Meeting notes\n' * 500).encode()
csv_  = ('id,value\n' + '\n'.join(f'{i},{i*i}' for i in range(2000))).encode()

samples = [('report.pdf', pdf), ('report_copy.pdf', pdf), ('Copy of report.pdf', pdf),
           ('photo1.jpg', photo), ('photo1_backup.jpg', photo),
           ('notes.txt', text), ('data.csv', csv_), ('data_old.csv', csv_),
           ('lecture_video.mp4', os.urandom(1_000_000))]

print('UPLOADING')
for name, data in samples:
    s, r = call('upload', file_name=name, file_b64=base64.b64encode(data).decode())
    flag = f"DUPLICATE of {r['duplicate_of']}" if r['duplicate_of'] else 'unique'
    print(f"  {name:20} {r['size_readable']:>10}  sha256 {r['sha256'][:12]}...  -> {flag}")

st = call('overview')[1]['stats']
print(f"\nBEFORE: {st['total_files']} files, {st['total_readable']} used, {st['duplicate_files']} duplicates wasting {st['wasted_readable']} ({st['wasted_pct']}%)")

s, prev = call('cleanup', strategy='oldest', dry_run=True)
print(f"\nPREVIEW: {prev['files']} files would move to trash, freeing {prev['saved_readable']}")
for p in prev['plan']:
    print(f"  - {p['name']:20} {p['size_readable']:>10}   (keeping: {p['keeping']})")

s, done = call('cleanup', strategy='oldest', dry_run=False)
st = call('overview')[1]['stats']
print(f"\nAFTER CLEANUP: {st['total_files']} files, {st['total_readable']} used, {st['duplicate_files']} duplicates | in trash: {st['trash_files']} ({st['trash_readable']})")

# safety rule: you can never trash every copy of a file
groups = call('overview')[1]
print('\nS3 now holds:', sorted(o['Key'].split('/')[0] for o in s3.list_objects_v2(Bucket=BUCKET)['Contents']))

# restore everything so the web app starts with duplicates to clean
ids = [t['file_id'] for t in call('overview')[1]['trash']]
print('Restored', call('restore', file_ids=ids)[1]['restored'], 'files (so you can try the cleanup in the web app)')

UPLOADING
  report.pdf             58.60 KB  sha256 3e3686263f9f...  -> unique
  report_copy.pdf        58.60 KB  sha256 3e3686263f9f...  -> DUPLICATE of report.pdf
  Copy of report.pdf     58.60 KB  sha256 3e3686263f9f...  -> DUPLICATE of report.pdf
  photo1.jpg            195.31 KB  sha256 fdb0853ab410...  -> unique
  photo1_backup.jpg     195.31 KB  sha256 fdb0853ab410...  -> DUPLICATE of photo1.jpg
  notes.txt               6.84 KB  sha256 fc84e731c627...  -> unique
  data.csv               22.89 KB  sha256 9b8441349c73...  -> unique
  data_old.csv           22.89 KB  sha256 9b8441349c73...  -> DUPLICATE of data.csv
  lecture_video.mp4     976.56 KB  sha256 9a0b95a5cbe0...  -> unique

BEFORE: 9 files, 1.56 MB used, 4 duplicates wasting 335.40 KB (21.0%)

PREVIEW: 4 files would move to trash, freeing 335.40 KB
  - data_old.csv           22.89 KB   (keeping: data.csv)
  - report_copy.pdf        58.60 KB   (keeping: report.pdf)
  - Copy of report.pdf     58.60 KB   (keeping: report.pd

In [ ]:
from werkzeug.serving import make_server

try:                      # allow re-running this cell
    server.shutdown()
except Exception:
    pass

server = make_server('0.0.0.0', 5000, app, threaded=True)
threading.Thread(target=server.serve_forever, daemon=True).start()
print('Flask server running on port 5000')

try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(5000, height=1400)   # shows the website below this cell
    # output.serve_kernel_port_as_window(5000)              # or open it in a new browser tab
except Exception:
    print('Open http://127.0.0.1:5000 in your browser')

Flask server running on port 5000


<IPython.core.display.Javascript object>